# Research V2: strict five-fold Jina-v2 boundary reranker

This notebook runs only the preregistered `CE -> parent max -> deterministic Top-5` hypothesis. It does not tune candidate retrieval, blend renderers, train LR, or upload a submission. By default it stops after writing the Fold-0 `PILOT_REPORT.json`, even when the pilot passes; full five-fold requires a separate manual post-review approval flag.

**First-time Kaggle setup.** Create one private Kaggle Dataset named `research-v2-jina-boundary` from the staged bundle. Attach it as an Input. In Notebook Settings choose GPU **T4 x2** and Persistence **Files only**. In Dependency Manager pin `transformers==4.40.2`, `peft==0.11.1`, `accelerate==0.30.1`, `safetensors==0.4.3`, and `einops==0.8.0`; do not upgrade NumPy/SciPy in-kernel. Outputs and resumable checkpoints are written under `/kaggle/working/research_v2_jina_boundary`. Save a Notebook Version before stopping. Stop safely with **Session -> Stop Session** only after both child processes have exited and the final manifest is present.

The two T4s are separate ~16 GiB devices. The notebook runs one fold process per GPU. Gradient accumulation increases effective batch size without changing per-microbatch VRAM and does not make gradients weaker: losses are divided by the accumulation count so the accumulated update is the mean gradient.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, time
import torch, transformers, peft
INPUT = Path('/kaggle/input/research-v2-jina-boundary')
WORK = Path('/kaggle/working/research_v2_jina_boundary')
WORK.mkdir(parents=True, exist_ok=True)
assert transformers.__version__.startswith('4.40.'), transformers.__version__
assert peft.__version__.startswith('0.11.'), peft.__version__
assert torch.cuda.device_count() == 2, f'Expected T4 x2, got {torch.cuda.device_count()}'
print([(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/2**30,2)) for i in range(2)])
print('input', INPUT, 'work', WORK)

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8<<20),b''): h.update(b)
    return h.hexdigest()
manifest=json.loads((INPUT/'KAGGLE_INPUT_MANIFEST.json').read_text())
for rel, expected in manifest['files_sha256'].items():
    actual=sha256(INPUT/rel)
    assert actual==expected, (rel,actual,expected)
assert manifest['v2_folds_sha256']=='94ad5c6d5e582ced5eec8d2c3c15f938454c17e713614391091e72abea9aba19'
print('INPUT CONTRACT PASS', len(manifest['files_sha256']), 'files')

## Bounded strict-OOF pilot

Fold 0 is trained only from folds 1-4 (with cross-fold exact/near-duplicate exclusions) and scored only on fold 0. The fixed pilot is 1,000 training groups and at most 400 optimizer microsteps. PASS requires Recall delta >= 0.003, improved boundary-pair accuracy, wins > losses, and multi-gold delta >= -0.005. Failure stops the family; no grid is attempted.

In [ ]:
PY=sys.executable
RUNNER=INPUT/'jina_v2_boundary_train.py'
MODEL=INPUT/'jina-reranker-v2-base-multilingual'
GROUPS=INPUT/'V2_BOUNDARY_GROUPS.jsonl'
GROUP_MANIFEST=INPUT/'V2_BOUNDARY_GROUPS_MANIFEST.json'
POOL=INPUT/'V2_CANDIDATE_POOL.jsonl'
CONTEXTS=INPUT/'V2_CONTEXTS.jsonl'
BASE_DB=INPUT/'evidence_ab_scores.sqlite'
renderer=manifest['renderer']
probe=WORK/'resume_probe'
probe_env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', HF_MODULES_CACHE=str(WORK/'hf_modules_gpu0'))
probe_common=[PY,str(RUNNER),'train','--model',str(MODEL),'--groups',str(GROUPS),'--groups-manifest',str(GROUP_MANIFEST),'--fold','fold_0','--microbatch','1','--accumulation','1','--max-train-groups','2','--eval-steps','1']
subprocess.run(probe_common+['--output',str(probe/'continuous'),'--max-steps','2'],env=probe_env,check=True)
subprocess.run(probe_common+['--output',str(probe/'first'),'--max-steps','1'],env=probe_env,check=True)
subprocess.run(probe_common+['--output',str(probe/'resumed'),'--max-steps','2','--resume',str(probe/'first'/'checkpoint-000001')],env=probe_env,check=True)
from safetensors.torch import load_file
a=load_file(probe/'continuous'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors')
b=load_file(probe/'resumed'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors')
resume_parity=(a.keys()==b.keys() and all(torch.equal(a[k],b[k]) for k in a))
assert resume_parity, 'Checkpoint/resume tensor parity failed'
pilot=WORK/'pilot_fold_0'
env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', HF_MODULES_CACHE=str(WORK/'hf_modules_gpu0'))
train_cmd=[PY,str(RUNNER),'train','--model',str(MODEL),'--groups',str(GROUPS),'--groups-manifest',str(GROUP_MANIFEST),'--fold','fold_0','--output',str(pilot),'--microbatch','2','--accumulation','8','--max-train-groups','1000','--max-steps','400','--eval-steps','96']
subprocess.run(train_cmd,env=env,check=True)
score_cmd=[PY,str(RUNNER),'score','--model',str(MODEL),'--groups',str(GROUPS),'--fold','fold_0','--adapter',str(pilot/'best'/'adapter'),'--pool',str(POOL),'--contexts-pack',str(CONTEXTS),'--base-score-db',str(BASE_DB),'--renderer',renderer,'--output',str(pilot/'score'),'--score-batch','16']
subprocess.run(score_cmd,env=env,check=True)
m=json.loads((pilot/'score'/'fold_0_METRICS.json').read_text())
multi_delta=m['multi_gold']['ft_recall']-m['multi_gold']['base_recall']
gate_checks={'delta_recall_at_5_gte_0_003':m['delta_recall_at_5']>=0.003,'boundary_pair_accuracy_improved':m['boundary_pair_accuracy']['ft']>m['boundary_pair_accuracy']['base'],'wins_exceed_losses':m['wins']>m['losses'],'multi_gold_delta_gte_minus_0_005':multi_delta>=-0.005}
train_manifest=json.loads((pilot/'TRAINING_MANIFEST.json').read_text())
gate_checks['peak_allocated_vram_mib_lt_15000']=train_manifest['peak_allocated_mib']<15000
gate_checks['resume_tensor_parity']=resume_parity
pilot_pass=all(gate_checks.values())
prediction_path=pilot/'score'/'fold_0_PREDICTIONS.jsonl'
metrics_path=pilot/'score'/'fold_0_METRICS.json'
continuous_adapter=probe/'continuous'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors'
resumed_adapter=probe/'resumed'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors'
pilot_report={'schema_version':'dsc2026.research_v2.jina_boundary_pilot_report.v1','status':'PASS' if pilot_pass else 'FAIL_STOP_FAMILY','next_action':'STOP_FOR_USER_REVIEW' if pilot_pass else 'STOP_FAMILY_WITHOUT_GRID','full_five_fold_launched':False,'immutable_contract':{'fold':'fold_0','candidate_pool_sha256':sha256(POOL),'groups_sha256':sha256(GROUPS),'groups_manifest_sha256':sha256(GROUP_MANIFEST),'renderer':renderer,'base_model_directory':str(MODEL),'max_length':512,'parent_aggregation':'max','output':'deterministic Top-5; no LR, rules, or fusion'},'pilot_config':{'max_train_groups':1000,'max_steps':400,'microbatch_parent_pairs':2,'gradient_accumulation':8,'eval_steps':96},'preregistered_gate':{'delta_recall_at_5_min':0.003,'boundary_pair_accuracy_must_improve':True,'wins_must_exceed_losses':True,'multi_gold_delta_min':-0.005,'peak_allocated_vram_mib_max_exclusive':15000,'resume_tensor_parity_required':True},'gate_checks':gate_checks,'multi_gold_delta':multi_delta,'metrics':m,'training':train_manifest,'resume_probe':{'tensor_parity':resume_parity,'continuous_adapter_sha256':sha256(continuous_adapter),'resumed_adapter_sha256':sha256(resumed_adapter)},'environment':{'python':sys.version,'torch':torch.__version__,'transformers':transformers.__version__,'peft':peft.__version__,'gpus':[{'index':i,'name':torch.cuda.get_device_name(i),'total_memory_bytes':torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]},'artifacts':{'input_manifest_sha256':sha256(INPUT/'KAGGLE_INPUT_MANIFEST.json'),'runner_sha256':sha256(RUNNER),'fold0_predictions':str(prediction_path),'fold0_predictions_sha256':sha256(prediction_path),'fold0_metrics':str(metrics_path),'fold0_metrics_sha256':sha256(metrics_path),'training_manifest_sha256':sha256(pilot/'TRAINING_MANIFEST.json')}}
(WORK/'PILOT_REPORT.json').write_text(json.dumps(pilot_report,indent=2))
(WORK/'PILOT_GATE.json').write_text(json.dumps({'status':pilot_report['status'],'report':'PILOT_REPORT.json','full_five_fold_launched':False},indent=2))
print(json.dumps(pilot_report,indent=2))
print('DEFAULT STOP: review PILOT_REPORT.json before enabling the separate full-five-fold cell.')

## Full five-fold campaign (manual launch only after pilot review)

The notebook stops after Fold-0 even when the pilot passes. Review `PILOT_REPORT.json` first, then deliberately set `USER_REVIEWED_PILOT_AND_APPROVES_FULL=True` in the next cell to launch full five-fold. Leaving the flag at its default `False` raises the expected fail-closed stop before any fold process is created. Each fold starts from the same frozen base and has its own adapter. Two child processes run in parallel, one per T4. If a session ends, set `RESUME` entries below to the last complete `checkpoint-*` directories and rerun that wave; the runner restores adapter, optimizer, and step.

In [ ]:
USER_REVIEWED_PILOT_AND_APPROVES_FULL=False  # change manually only after reviewing PILOT_REPORT.json
if not USER_REVIEWED_PILOT_AND_APPROVES_FULL:
    raise RuntimeError('EXPECTED DEFAULT STOP: full five-fold requires explicit user review approval')
pilot_report=json.loads((WORK/'PILOT_REPORT.json').read_text())
assert pilot_report['status']=='PASS', 'Full five-fold is forbidden because the fixed pilot did not pass.'
assert pilot_report['full_five_fold_launched'] is False
pilot_report['full_five_fold_launched']=True
pilot_report['full_launch_authorization']='USER_REVIEWED_PILOT_AND_APPROVES_FULL=True'
(WORK/'PILOT_REPORT.json').write_text(json.dumps(pilot_report,indent=2))
FULL=WORK/'full'; FULL.mkdir(exist_ok=True)
RESUME={}  # e.g. {'fold_2': FULL/'fold_2'/'checkpoint-001000'}
pilot_train=json.loads((pilot/'TRAINING_MANIFEST.json').read_text())
microbatch=4 if pilot_train['peak_allocated_mib']<7500 else 2
accumulation=4 if microbatch==4 else 8
def launch_train(fold,gpu):
    out=FULL/fold; out.mkdir(exist_ok=True)
    cmd=[PY,str(RUNNER),'train','--model',str(MODEL),'--groups',str(GROUPS),'--groups-manifest',str(GROUP_MANIFEST),'--fold',fold,'--output',str(out),'--microbatch',str(microbatch),'--accumulation',str(accumulation),'--eval-steps','200']
    if fold in RESUME: cmd += ['--resume',str(RESUME[fold])]
    log=open(out/'train.log','a',buffering=1)
    return subprocess.Popen(cmd,env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),TOKENIZERS_PARALLELISM='false',HF_MODULES_CACHE=str(WORK/f'hf_modules_gpu{gpu}')),stdout=log,stderr=subprocess.STDOUT),log
for wave in [('fold_0','fold_1'),('fold_2','fold_3'),('fold_4',)]:
    jobs=[launch_train(f,g) for g,f in enumerate(wave)]
    codes=[p.wait() for p,_ in jobs]
    for _,log in jobs: log.close()
    assert all(c==0 for c in codes),(wave,codes)
print('all fold checkpoints complete')

In [ ]:
if not globals().get('USER_REVIEWED_PILOT_AND_APPROVES_FULL',False):
    raise RuntimeError('EXPECTED DEFAULT STOP: full five-fold scoring also requires explicit user review approval')
assert json.loads((WORK/'PILOT_REPORT.json').read_text())['full_five_fold_launched'] is True
def launch_score(fold,gpu):
    out=FULL/fold/'score'; out.mkdir(exist_ok=True)
    cmd=[PY,str(RUNNER),'score','--model',str(MODEL),'--groups',str(GROUPS),'--fold',fold,'--adapter',str(FULL/fold/'best'/'adapter'),'--pool',str(POOL),'--contexts-pack',str(CONTEXTS),'--base-score-db',str(BASE_DB),'--renderer',renderer,'--output',str(out),'--score-batch','16']
    log=open(out/'score.log','a',buffering=1)
    return subprocess.Popen(cmd,env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),TOKENIZERS_PARALLELISM='false',HF_MODULES_CACHE=str(WORK/f'hf_modules_gpu{gpu}')),stdout=log,stderr=subprocess.STDOUT),log
for wave in [('fold_0','fold_1'),('fold_2','fold_3'),('fold_4',)]:
    jobs=[launch_score(f,g) for g,f in enumerate(wave)]
    codes=[p.wait() for p,_ in jobs]
    for _,log in jobs: log.close()
    assert all(c==0 for c in codes),(wave,codes)
metrics=[json.loads((FULL/f/'score'/f'{f}_METRICS.json').read_text()) for f in [f'fold_{i}' for i in range(5)]]
n=sum(m['queries'] for m in metrics)
def pooled(field): return sum(m[field]*m['queries'] for m in metrics)/n
def pooled_slice(name,field):
    z=sum(m[name]['queries'] for m in metrics); return sum(m[name][field]*m[name]['queries'] for m in metrics)/z
summary={'status':'COMPLETE_STRICT_FIVE_FOLD','queries':n,'pooled_base_recall_at_5':pooled('base_recall_at_5'),'pooled_ft_recall_at_5':pooled('ft_recall_at_5'),'pooled_base_precision_at_5':pooled('base_precision_at_5'),'pooled_ft_precision_at_5':pooled('ft_precision_at_5'),'single_gold':{'base':pooled_slice('single_gold','base_recall'),'ft':pooled_slice('single_gold','ft_recall')},'multi_gold':{'base':pooled_slice('multi_gold','base_recall'),'ft':pooled_slice('multi_gold','ft_recall')},'wins':sum(m['wins'] for m in metrics),'losses':sum(m['losses'] for m in metrics),'ties':sum(m['ties'] for m in metrics),'changed_top5_sets':sum(m['changed_top5_sets'] for m in metrics),'peak_allocated_mib_per_fold':{m['fold']:m['peak_allocated_mib'] for m in metrics},'runtime_seconds_per_fold':{m['fold']:m['runtime_seconds'] for m in metrics},'per_fold':metrics}
summary['pooled_delta_recall_at_5']=summary['pooled_ft_recall_at_5']-summary['pooled_base_recall_at_5']
(WORK/'FULL_OOF_SUMMARY.json').write_text(json.dumps(summary,indent=2))
print(json.dumps({k:v for k,v in summary.items() if k!='per_fold'},indent=2))